# PTQ Edge Benchmark — Analysis & Visualization

Load experiment CSV/JSON results and produce publication-ready figures.

**Paper**: *Comparative Analysis of Post-Training Quantization Methods for LLM Inference on Edge Devices*

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.2)
RESULTS_DIR = Path('../results')

# Load benchmark results
dfs = []
for csv_file in RESULTS_DIR.glob('benchmark_*.csv'):
    df = pd.read_csv(csv_file)
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
print(f'Loaded {len(df)} benchmark entries')
df.head()

## 1. Latency Comparison across Methods & Hardware

In [ ]:
if not df.empty:
    fig, axes = plt.subplots(1, len(df['hardware'].unique()), figsize=(14, 5), sharey=False)
    if not hasattr(axes, '__iter__'):
        axes = [axes]

    for ax, hw in zip(axes, sorted(df['hardware'].unique())):
        sub = df[df['hardware'] == hw].sort_values('latency_ms')
        colors = sns.color_palette('muted', len(sub))
        bars = ax.bar(sub['method'], sub['latency_ms'], color=colors, edgecolor='black', linewidth=0.6)
        ax.bar_label(bars, fmt='%.0f ms', padding=3, fontsize=10)
        ax.set_title(f'Latency — {hw}', fontweight='bold')
        ax.set_ylabel('Latency (ms)')
        ax.set_xlabel('Quantization Method')

    plt.suptitle('Inference Latency by Method and Hardware', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'fig_latency_comparison.pdf', dpi=300, bbox_inches='tight')
    plt.show()
    print('Saved: fig_latency_comparison.pdf')

## 2. Memory vs Accuracy (Perplexity) Trade-off

In [ ]:
if not df.empty and df['perplexity'].notna().any():
    fig, ax = plt.subplots(figsize=(8, 5))
    markers = ['o', 's', '^', 'D', 'P']
    for i, (method, g) in enumerate(df.groupby('method')):
        ax.scatter(g['peak_memory_mb'], g['perplexity'],
                   label=method, s=120, marker=markers[i % len(markers)], zorder=3)
    ax.set_xlabel('Peak Memory (MB)')
    ax.set_ylabel('Perplexity ↓ (lower is better)')
    ax.set_title('Memory–Accuracy Trade-off by Quantization Method', fontweight='bold')
    ax.legend(title='Method')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'fig_memory_accuracy_tradeoff.pdf', dpi=300, bbox_inches='tight')
    plt.show()
    print('Saved: fig_memory_accuracy_tradeoff.pdf')

## 3. Throughput Heatmap (Model × Method)

In [ ]:
if not df.empty:
    pivot = df.pivot_table(values='throughput_tok_per_sec',
                           index='model_name', columns='method', aggfunc='mean')
    fig, ax = plt.subplots(figsize=(10, max(4, len(pivot) * 0.8)))
    sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd',
                linewidths=0.5, ax=ax, cbar_kws={'label': 'tok/s'})
    ax.set_title('Throughput (tok/s) — Model × Method', fontweight='bold')
    ax.set_xlabel('Method')
    ax.set_ylabel('Model')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'fig_throughput_heatmap.pdf', dpi=300, bbox_inches='tight')
    plt.show()

## 4. Summary Table (LaTeX-ready)

In [ ]:
if not df.empty:
    summary = df.groupby(['hardware', 'method']).agg(
        latency_ms=('latency_ms', 'mean'),
        throughput=('throughput_tok_per_sec', 'mean'),
        memory_mb=('peak_memory_mb', 'mean'),
        perplexity=('perplexity', 'mean'),
    ).round(2)

    print(summary.to_string())
    print('\n--- LaTeX ---')
    print(summary.to_latex(float_format='%.2f', caption='Benchmark Results Summary',
                           label='tab:results'))